# CA Cities Boundaries


### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
import climakitae as ck

from climakitae.core.data_load import load

import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from climakitae.new_core.data_access.boundaries import Boundaries
import intake

### Step 1: Filter out offshore components from coastal cities

In [2]:
boundaries_shp = gpd.read_file(
    "California_Cities_and_Identifiers_Blue_Version_view_-3024944392275494958.zip"
)

In [5]:
boundaries_shp.shape

(549, 17)

filter out offshore elements

In [16]:
boundaries_shp_filtered = boundaries_shp[boundaries_shp['OFFSHORE'].isna()]

In [19]:
boundaries_shp_filtered['OFFSHORE'].unique()

array([None], dtype=object)

### Step 2: Convert boundaries shapefile to parquet

In [21]:
boundaries_shp_filtered.to_parquet("CA_cities_boundaries_filtered.parquet")

### Step 3: Update new core boundaries file

In [2]:
catalog = intake.open_catalog('boundaries.yaml')
boundaries = Boundaries(catalog)
# Get all boundary options for UI population
boundary_options = boundaries.boundary_dict()

In [3]:
# Access specific boundary data (loaded lazily)
ca_cities = boundaries._ca_cities
ca_counties= boundaries._ca_counties

In [8]:
ca_cities.shape

(549, 17)

In [13]:
ca_cities[ca_cities['CDT_NAME_S']=="Fresno"]

,CDTFA_COPR,CDTFA_CITY,CDTFA_COUN,CENSUS_PLA,CENSUS_GEO,CENSUS_P_1,GNIS_PLACE,GNIS_ID,CDT_CITY_A,CDT_COUNTY,PRIMARY_DO,CENSUS_POP,CDT_NAME_S,OFFSHORE,AREA_SQMI,GlobalID,geometry
150,10005,Fresno,Fresno County,Fresno,0627000,City,City of Fresno,2410546,FRES,FRE,None,0,Fresno,None,117.258242,70a4c648-5a1e-4b3a-bd3b-838ae518bbe2,"MULTIPOLYGON (((24035.889 -124909.833, 24037.8..."


In [16]:
ca_counties[ca_counties["NAME"] == "Los Angeles County"]

,NAME,geometry
5,Los Angeles County,"MULTIPOLYGON (((-118.02878 33.87332, -118.0252..."


In [19]:
ca_counties["NAME"].duplicated().sum()

np.int64(0)

In [22]:
ca_cities["CDT_NAME_S"].duplicated().sum()

np.int64(66)

In [14]:
boundaries.get_memory_usage()

{'states': np.int64(9579),
 'ca_counties': np.int64(4609),
 'ca_cities': np.int64(411663),
 'ca_watersheds': np.int64(30345),
 'ca_utilities': np.int64(8900),
 'ca_forecast_zones': np.int64(5933),
 'ca_electric_balancing_areas': np.int64(698),
 'ca_census_tracts': np.int64(5378428),
 'total_bytes': np.int64(5850155),
 'total_human': '5.6 MB',
 'loaded_datasets': 8,
 'cached_lookups': 8}

In [9]:
boundary_options['CA cities']

{'Adelanto': 0,
 'Agoura Hills': 1,
 'Alameda': 2,
 'Albany': 484,
 'Alhambra': 4,
 'Aliso Viejo': 5,
 'Alturas': 6,
 'Amador City': 7,
 'American Canyon': 8,
 'Anaheim': 9,
 'Anderson': 10,
 'Angels': 11,
 'Antioch': 12,
 'Apple Valley': 13,
 'Arcadia': 14,
 'Arcata': 15,
 'Arroyo Grande': 16,
 'Artesia': 17,
 'Arvin': 18,
 'Atascadero': 19,
 'Atherton': 20,
 'Atwater': 21,
 'Auburn': 22,
 'Avalon': 23,
 'Avenal': 24,
 'Azusa': 25,
 'Bakersfield': 26,
 'Baldwin Park': 27,
 'Banning': 28,
 'Barstow': 29,
 'Beaumont': 30,
 'Bell': 31,
 'Bell Gardens': 32,
 'Bellflower': 33,
 'Belmont': 34,
 'Belvedere': 486,
 'Benicia': 487,
 'Berkeley': 488,
 'Beverly Hills': 38,
 'Big Bear Lake': 39,
 'Biggs': 40,
 'Bishop': 41,
 'Blue Lake': 42,
 'Blythe': 43,
 'Bradbury': 44,
 'Brawley': 45,
 'Brea': 46,
 'Brentwood': 47,
 'Brisbane': 48,
 'Buellton': 49,
 'Buena Park': 50,
 'Burbank': 51,
 'Burlingame': 490,
 'Calabasas': 53,
 'Calexico': 54,
 'California City': 55,
 'Calimesa': 56,
 'Calipatria': 

#### Step 2: Update clip processor

In [ ]:
wl_clipped_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d03")
    .variable_id("t2")  # Temperature at 2 meters
    .processes(
        {
            "warming_level": {
                "warming_levels": [1.5, 2.0, 3.0],  # Warming levels in °C
            },
            "clip": (34.05, -118.25),  # Clip to specified coordinates
        }
    )
    .get()
)

wl_clipped_data